In [11]:
import os
from PIL import Image
import matplotlib.pyplot as plt

emoji_dir = "data/G-emojis"
emoji_files = os.listdir(emoji_dir)

print("Number of emojis:", len(emoji_files))
emoji_files[:5]


Number of emojis: 3793


['1f469-1f3fc-200d-1f9b2.png',
 '1f97f.png',
 '1f9a7.png',
 '1f647-1f3fd-200d-2642-fe0f.png',
 '1f6a3-1f3fc.png']

In [12]:
import numpy as np
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image

#pre trained CNN model 
model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

In [19]:
#converts one emoji image into a feature vector
def extract_features(img_path, target_size=(224, 224)):
    """Convert one emoji image into a feature vector"""
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    features = model.predict(img_array, verbose=0)
    return features.flatten()

In [18]:
# gets the features from all the emojis
features = []
valid_files = []

print("Extracting features from all emojis...")

for i, emoji_file in enumerate(emoji_files):
    # updates after every 500 emojis so we get a progress update
    if i % 500 == 0:
        print(f"Processing emoji {i}/{len(emoji_files)}...")
    
    try:
        img_path = os.path.join(emoji_dir, emoji_file)
        feat = extract_features(img_path)
        features.append(feat)
        valid_files.append(emoji_file)
    except Exception as e:
        # skips bad images
        pass

features = np.array(features)
print(f"\nExtracted features from {len(features)} emojis")
print(f"Feature matrix shape: {features.shape}")

Extracting features from all emojis
Processing emoji 0/3793...
Processing emoji 500/3793...
Processing emoji 1000/3793...
Processing emoji 1500/3793...
Processing emoji 2000/3793...
Processing emoji 2500/3793...
Processing emoji 3000/3793...
Processing emoji 3500/3793...

Extracted features from 3793 emojis
Feature matrix shape: (3793, 1280)


In [24]:
from sklearn.cluster import KMeans

# clusters the emojis into groups
n_clusters = 15
print(f"Clustering {len(features)} emojis into {n_clusters} groups...\n")

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(features)

print(f"✅ Clustering complete!")
print(f"\nEmojis per cluster:")
for i in range(n_clusters):
    count = np.sum(labels == i)
    print(f"  Cluster {i}: {count} emojis")

Clustering 3793 emojis into 15 groups...

✅ Clustering complete!

Emojis per cluster:
  Cluster 0: 203 emojis
  Cluster 1: 107 emojis
  Cluster 2: 198 emojis
  Cluster 3: 108 emojis
  Cluster 4: 509 emojis
  Cluster 5: 208 emojis
  Cluster 6: 358 emojis
  Cluster 7: 199 emojis
  Cluster 8: 334 emojis
  Cluster 9: 104 emojis
  Cluster 10: 307 emojis
  Cluster 11: 244 emojis
  Cluster 12: 213 emojis
  Cluster 13: 307 emojis
  Cluster 14: 394 emojis
